# 01 — Data Exploration
**PCA Macro Slowdown Indicator**  
Sudhan Adithya · Kavin Dhanasekar

This notebook covers:
- Loading and visualizing raw FRED series
- Missing value audit
- Stationarity / transformation review
- Pairwise correlation matrix of raw macro variables

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
load_dotenv('../.env')

import config
from data.fetch_data import get_fred_client, fetch_all_macro, fetch_all_financial, fetch_reference_series
from data.align import align_macro_to_monthly, align_financial_to_monthly, merge_panel, trim_to_overlap

print('Setup complete')

In [ ]:
# --- Fetch data (skip if already done) ---
fred = get_fred_client()
macro_raw = fetch_all_macro(fred)
fin_raw   = fetch_all_financial(fred)
ref_raw   = fetch_reference_series(fred)

In [ ]:
# --- Align and merge ---
macro_m = align_macro_to_monthly(macro_raw)
fin_m   = align_financial_to_monthly(fin_raw)
panel   = merge_panel(macro_m, fin_m, ref_raw)
panel   = trim_to_overlap(panel)
panel.tail()

In [ ]:
# --- Missing value heatmap ---
fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(panel.isnull().T, cbar=False, yticklabels=True, ax=ax, cmap='Reds')
ax.set_title('Missing Values by Column', fontsize=12)
plt.tight_layout()
plt.show()
print('Missing counts:\n', panel.isnull().sum())

In [ ]:
# --- Plot all macro series (raw levels) ---
macro_cols = [c for c in config.MACRO_PCA_COLS if c in panel.columns]
fig, axes = plt.subplots(len(macro_cols), 1, figsize=(14, 2.5 * len(macro_cols)), sharex=True)
for ax, col in zip(axes, macro_cols):
    ax.plot(panel.index, panel[col], linewidth=1.2)
    ax.set_title(col, fontsize=10)
    ax.grid(True, alpha=0.3)
plt.suptitle('Raw Macro Series (Levels)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# --- Correlation matrix (raw levels) ---
corr = panel[macro_cols].corr()
fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax,
            linewidths=0.5, square=True)
ax.set_title('Correlation Matrix — Raw Macro Levels', fontsize=12)
plt.tight_layout()
plt.show()